# Part 1: Data exploration, pre-processing charts and cleaning
Role: Execution lead
 
Depends on variables from earlier parts (listed below); not standalone. Run main.ipynb for the full pipeline.

**Needs:** nothing.

**Hands over:** `df`, `X`, `y`.


## Setup (repeated from main.ipynb)

This cell finds the project root and the dataset path. It works whether Jupyter starts in `notebooks/` or in the project root.

In [ ]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Data").is_dir())
DATA_PATH = ROOT / "Data" / "Life Expectancy Data.csv"


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


## Part 1: Data exploration, pre-processing charts and cleaning (Execution lead)

The first function reads the CSV and cleans the column names. The raw file has extra spaces at the ends of names and double spaces inside some names, so both are fixed before any analysis. We then call it on the dataset path and define the target column once.


In [ ]:
def load_and_clean(path):
    """Read the CSV and clean the column names."""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)
    return df


In [ ]:
TARGET = "Life expectancy"

df = load_and_clean(DATA_PATH)
assert TARGET in df.columns

print("Shape:", df.shape)


Next we inspect the data: its size, the first rows, column types, summary statistics, and how many values are missing in each column.

In [ ]:
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
df.isna().sum()


The `plot_eda` function draws the charts we need before dropping anything. It shows the target distribution, missing values, correlations between numeric columns, and how three important features relate to life expectancy.

In [ ]:
def plot_eda(df):
    """Draw the main exploratory charts for the dataset."""
    plt.figure(figsize=(8, 5))
    plt.hist(df[TARGET].dropna(), bins=30)
    plt.title("Distribution of life expectancy")
    plt.xlabel("Life expectancy (years)")
    plt.ylabel("Number of rows")
    plt.tight_layout()
    plt.show()

    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    plt.figure(figsize=(10, 6))
    missing.plot(kind="bar")
    plt.title("Missing values per column")
    plt.xlabel("Column")
    plt.ylabel("Number of missing values")
    plt.tight_layout()
    plt.show()

    correlation = df.corr(numeric_only=True)
    plt.figure(figsize=(16, 12))
    sns.heatmap(correlation, cmap="coolwarm", center=0)
    plt.title("Correlation heatmap of numeric columns")
    plt.xlabel("Column")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.show()

    scatter_columns = ["Schooling", "Income composition of resources", "Adult Mortality"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, column in zip(axes, scatter_columns):
        ax.scatter(df[column], df[TARGET], alpha=0.4, s=10)
        ax.set_title(f"{column} vs life expectancy")
        ax.set_xlabel(column)
        ax.set_ylabel("Life expectancy (years)")
    plt.tight_layout()
    plt.show()


In [ ]:
plot_eda(df)


We now remove columns and rows that would hurt the model or cannot be used. The reasons are kept in a dictionary so they can be printed and checked.

The redundant columns are kept in one list.

In [ ]:
REDUNDANT_COLUMNS = ["infant deaths", "thinness 5-9 years", "percentage expenditure"]


`Country` is an identifier, rows without the target cannot be used, and the redundant columns duplicate information already in the dataset. The dictionary records one reason for each drop.

In [ ]:
DROP_REASONS = {
    "Country": "Identifier with about 193 unique values; one-hot encoding it would add hundreds of columns.",
    "Rows with missing Life expectancy": "The target cannot be imputed, so these rows are removed.",
    "infant deaths": "Almost the same information as under-five deaths.",
    "thinness 5-9 years": "Almost the same information as thinness 1-19 years.",
    "percentage expenditure": "Almost the same information as GDP."
}

df = df.drop(columns=["Country"] + REDUNDANT_COLUMNS)
df = df.dropna(subset=[TARGET]).reset_index(drop=True)

print("Dropped columns and rows:")
for name, reason in DROP_REASONS.items():
    print(f"- {name}: {reason}")
print("Shape after drops:", df.shape)

The cleaned dataframe is split into features `X` and target `y`. `Country` is already gone, and `TARGET` is the life expectancy column.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)


This part hands over `df`, `X` and `y`.